# 대학원 AI 논문 연구 도우미 v2.3 실습·검증

*각 연습 셀 다음에 **독립적인 assert 검증 셀**이 있습니다.*

노트북을 배포 폴더(`app.py`가 있는 곳)에서 실행하세요. OpenAI API나 공개데이터 API 키는 필요하지 않습니다.

## 실습 1. 정식 학위논문 참고문헌 표기

In [ ]:
from citation_styles import THESIS, complete_citation, format_reference

논문 = {"자료명/저자사항": "Amazon RDS 데이터베이스 디지털포렌식 조사 기법 연구 /김수용", "DB": "학위논문"}
서지 = complete_citation(논문, {"type": THESIS, "author": "김수용", "title": "Amazon RDS 데이터베이스 디지털포렌식 조사 기법 연구", "degree": "석사학위논문", "university": "고려대학교 대학원", "publication_date": "2026년 8월"})
정식참고문헌 = format_reference(서지)
print(정식참고문헌)

**검증 셀 1 — 제목 뒤 쉼표·대학·학위·연월을 확인합니다.**

In [ ]:
assert "연구”, 석사학위논문" in 정식참고문헌
assert "고려대학교 대학원" in 정식참고문헌
assert "2026년 8월" in 정식참고문헌
print("✅ 참고문헌 형식 검사 통과")

## 실습 2. AI 분석 요약을 선행연구 7개 항목으로 연결

In [ ]:
from research_workspace import source_dimensions, DIMENSIONS, INCOMPLETE
from prompt_templates import READING

가상분석 = {"mode": READING, "items": [
    {"key": "연구배경", "text": "클라우드 DB 환경의 포렌식 조사 필요성"},
    {"key": "연구방법·핵심기술", "text": "S3·Parquet 스냅숏 기반 분석"},
    {"key": "주요연구결과", "text": "포렌식 조사 절차를 제안"},
    {"key": "연구의의·활용", "text": "오프라인 분석 환경의 활용 가능성"},
    {"key": "연구한계·후속연구", "text": "삭제 데이터 복구에 한계"},
]}
항목 = source_dimensions(가상분석, "감사 로그가 없다면 무엇을 할까?")
for 이름, 설명 in 항목.items():
    print(이름, ':', 설명)

**검증 셀 2 — 결과와 기여도, 새로운 쟁점을 별도 열로 보관합니다.**

In [ ]:
assert len(항목) == 7
assert 항목["활용 기술"] == "S3·Parquet 스냅숏 기반 분석"
assert "사용자 생각, 미검증" in 항목["새로운 쟁점"]
print("✅ 7개 연구 항목 검사 통과")

## 실습 3. 선행연구 비교표 Excel 만들기

In [ ]:
from research_workspace import build_entry, literature_excel
from io import BytesIO
from openpyxl import load_workbook

기록 = build_entry(논문, 서지, 가상분석, "감사 로그가 없다면 무엇을 할까?")
엑셀바이트 = literature_excel([기록])
작업문서 = load_workbook(BytesIO(엑셀바이트))
print("시트:", 작업문서.sheetnames)
print("정식 참고문헌:", 작업문서["선행연구_비교표"]["A2"].value)

**검증 셀 3 — Excel이 열리고 정식 참고문헌이 유지되는지 확인합니다.**

In [ ]:
assert 엑셀바이트[:2] == b"PK"
assert "선행연구_비교표" in 작업문서.sheetnames
assert 작업문서["선행연구_비교표"]["A2"].value == 정식참고문헌
print("✅ Excel 비교표 검사 통과")

## 실습 4. JSON 프로젝트 백업 후 다시 읽기

In [ ]:
from research_workspace import export_snapshot, import_snapshot

백업 = export_snapshot([기록], {"research_interest": "데이터베이스 포렌식"})
읽어온기록, 프로젝트정보 = import_snapshot(백업)
print("복원 논문 수:", len(읽어온기록))
print("연구 주제:", 프로젝트정보["research_interest"])

**검증 셀 4 — 정식 참고문헌이 바뀌지 않았는지 확인합니다.**

In [ ]:
assert len(읽어온기록) == 1
assert 읽어온기록[0]["reference"] == 정식참고문헌
assert 프로젝트정보["research_interest"] == "데이터베이스 포렌식"
print("✅ 백업·복원 검사 통과")

## 실습 5. 나의 의문을 연구과제 검토 프롬프트로 변환

In [ ]:
from research_workspace import direction_prompt

연구프롬프트 = direction_prompt(읽어온기록, "데이터베이스 포렌식", "석사학위논문", "가상 실습용 로그", "6개월")
print(연구프롬프트[:600])

**검증 셀 5 — 연구 공백을 확정하지 않고 정식 참고문헌을 사용하도록 검사합니다.**

In [ ]:
assert 정식참고문헌 in 연구프롬프트
assert "검증 전 후보" in 연구프롬프트
assert "가상 실습용 로그" in 연구프롬프트
print("✅ 연구과제 검토 프롬프트 검사 통과")